# Datenexploration – Manufacturing-Datensatz
Dieses Notebook führt eine typische explorative Datenanalyse (EDA) für den Datensatz `data/raw/manufacturing.csv` durch – mit pandas, matplotlib und seaborn.

**Ziel des Datensatzes:** Vorhersage der Produktqualität (`Quality Rating`) anhand von Prozessparametern eines Fertigungsprozesses (Regressionsproblem).

| Spalte | Bedeutung |
|---|---|
| `Temperature (°C)` | Prozesstemperatur in Grad Celsius |
| `Pressure (kPa)` | Prozessdruck in Kilopascal |
| `Temperature x Pressure` | Interaktionsmerkmal: Temperatur · Druck |
| `Material Fusion Metric` | Abgeleitete Kennzahl für die Materialverschmelzung |
| `Material Transformation Metric` | Abgeleitete Kennzahl für die Materialumwandlung |
| `Quality Rating` | **Zielvariable**: Qualitätsbewertung (ca. 0–100) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## Daten laden
Die CSV-Datei wird eingelesen. Zur einfacheren Handhabung im Code werden die Spaltennamen in kurze englische `snake_case`-Namen umbenannt (ohne Sonderzeichen wie `°`).

In [ ]:
DATA_PATH = 'data/raw/manufacturing.csv'

df_raw = pd.read_csv(DATA_PATH)

# Map original column names to short, code-friendly names
column_mapping = {
    'Temperature (°C)': 'temperature',
    'Pressure (kPa)': 'pressure',
    'Temperature x Pressure': 'temp_x_pressure',
    'Material Fusion Metric': 'fusion_metric',
    'Material Transformation Metric': 'transformation_metric',
    'Quality Rating': 'quality_rating',
}
df = df_raw.rename(columns=column_mapping)

TARGET = 'quality_rating'
features = [col for col in df.columns if col != TARGET]

df.head()

## Grundlegender Überblick
Anzahl der Zeilen/Spalten, Datentypen und Speicherbedarf. Alle Spalten sollten numerisch (`float64`) sein.

In [ ]:
print(df.shape)
print(df.dtypes)
df.info()

Deskriptive Statistik (Mittelwert, Standardabweichung, Min/Max, Quartile) je Spalte.

**Beobachtungen:**
- Temperatur liegt zwischen ca. 100 °C und 300 °C, Druck zwischen ca. 5 kPa und 25 kPa – beide scheinen gleichverteilt gesampelt zu sein.
- Die abgeleiteten Metriken haben sehr unterschiedliche Größenordnungen (bis ca. 2,7 · 10⁷) → für viele Modelle ist eine **Skalierung** notwendig.
- Beim `quality_rating` liegt der Median nahe 100, der Mittelwert aber deutlich darunter → Hinweis auf eine **stark linksschiefe Verteilung**.

In [ ]:
df.describe().T

## Fehlende Werte und Duplikate
Prüfung, ob Werte fehlen oder identische Zeilen mehrfach vorkommen.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nNumber of duplicated rows: {df.duplicated().sum()}')

## Überprüfung der abgeleiteten Merkmale
Die Spalten `temp_x_pressure`, `fusion_metric` und `transformation_metric` sind offenbar aus Temperatur (T) und Druck (P) berechnet. Wir prüfen die folgenden Hypothesen:

- `temp_x_pressure` = T · P
- `fusion_metric` = T² + P³
- `transformation_metric` = T³ − P²

Ist die maximale absolute Abweichung praktisch 0 (nur Rundungsfehler), bestätigt sich die Formel. Das bedeutet: Der Datensatz enthält eigentlich **nur zwei unabhängige Eingangsgrößen** – alle anderen Merkmale sind deterministische Transformationen davon (relevant für Multikollinearität und Feature Engineering).

In [ ]:
temp = df['temperature']
pres = df['pressure']

# Candidate formulas for the derived columns
derived_formulas = {
    'temp_x_pressure': temp * pres,
    'fusion_metric': temp**2 + pres**3,
    'transformation_metric': temp**3 - pres**2,
}

for col, expected in derived_formulas.items():
    max_abs_diff = (df[col] - expected).abs().max()
    max_rel_diff = ((df[col] - expected).abs() / df[col].abs()).max()
    print(f'{col:<22} max abs diff: {max_abs_diff:.2e}   max rel diff: {max_rel_diff:.2e}')

## Verteilungsanalyse
Histogramme aller Spalten. Temperatur und Druck sind näherungsweise gleichverteilt; die abgeleiteten Metriken sind durch die Potenzen verzerrt (schief).

In [ ]:
df.hist(figsize=(12, 8), bins=40)
plt.tight_layout()
plt.show()

### Zielvariable `quality_rating`
Die Zielvariable verdient eine genauere Betrachtung: Der Großteil der Werte liegt sehr nahe bei 100, während eine kleinere Gruppe deutlich niedrigere Werte (bis hinunter zu 1) aufweist. Die logarithmische y-Achse macht diese seltenen niedrigen Werte sichtbar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df[TARGET], bins=60, ax=axes[0])
axes[0].set_title('Quality Rating distribution')

sns.histplot(df[TARGET], bins=60, ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title('Quality Rating distribution (log y-axis)')

plt.tight_layout()
plt.show()

# Share of samples above selected quality thresholds
for threshold in [99, 95, 90, 50]:
    share = (df[TARGET] > threshold).mean()
    print(f'Share with quality > {threshold}: {share:.1%}')
print(f'Skewness: {df[TARGET].skew():.2f}')

## Ausreißeranalyse
Boxplots zeigen Ausreißer nach der IQR-Regel (Punkte außerhalb von 1,5 · IQR). Da die Spalten sehr unterschiedliche Skalen haben, wird jede Spalte in einem eigenen Plot dargestellt.

**Hinweis:** Die „Ausreißer“ im `quality_rating` sind keine Messfehler, sondern genau die interessanten Fälle schlechter Qualität – sie sollten **nicht** entfernt werden.

In [ ]:
fig, axes = plt.subplots(1, len(df.columns), figsize=(16, 4))
for ax, col in zip(axes, df.columns):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

# Count outliers per column using the IQR rule
q1 = df.quantile(0.25)
q3 = df.quantile(0.75)
iqr = q3 - q1
outlier_mask = (df < q1 - 1.5 * iqr) | (df > q3 + 1.5 * iqr)
outlier_mask.sum()

## Korrelationsanalyse
Pearson-Korrelation (linear) und Spearman-Korrelation (monoton, rangbasiert). Da der Zusammenhang mit der Qualität stark nichtlinear ist, ist die Spearman-Korrelation hier aussagekräftiger.

**Erwartung:** Temperatur und die temperaturabhängigen Metriken korrelieren stark untereinander (Multikollinearität) und negativ mit der Qualität; der Druck hat kaum einen Einfluss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, method in zip(axes, ['pearson', 'spearman']):
    corr = df.corr(method=method)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'Correlation Matrix ({method})')

plt.tight_layout()
plt.show()

## Zusammenhang zwischen Merkmalen und Zielvariable
Streudiagramme jedes Merkmals gegen `quality_rating`. Hier wird sichtbar, dass die Qualität bis zu einer bestimmten Temperatur nahezu konstant bei 100 liegt und danach stark abfällt (nichtlinearer „Knick“).

In [ ]:
fig, axes = plt.subplots(1, len(features), figsize=(20, 4), sharey=True)
for ax, col in zip(axes, features):
    ax.scatter(df[col], df[TARGET], s=5, alpha=0.4)
    ax.set_xlabel(col)
axes[0].set_ylabel(TARGET)
plt.tight_layout()
plt.show()

Temperatur und Druck gemeinsam betrachtet, eingefärbt nach Qualität: So erkennt man, in welchem Bereich des Prozessfensters die Qualität einbricht.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(df['temperature'], df['pressure'], c=df[TARGET], cmap='viridis', s=8)
plt.colorbar(scatter, label=TARGET)
plt.xlabel('temperature')
plt.ylabel('pressure')
plt.title('Process window colored by Quality Rating')
plt.show()

## Pairplot
Paarweise Streudiagramme aller Merkmale. Da es keine kategoriale Spalte gibt, wird die Temperatur in vier Bereiche eingeteilt und als Farbe (`hue`) verwendet.

In [ ]:
# Bin temperature into ranges to use as a categorical hue
df_plot = df.copy()
df_plot['temp_range'] = pd.cut(df_plot['temperature'], bins=[100, 150, 200, 250, 300], include_lowest=True)

sns.pairplot(df_plot, hue='temp_range', plot_kws={'s': 8, 'alpha': 0.5})
plt.show()

## Gruppierte Statistiken
Kennzahlen der Qualität je Temperaturbereich bzw. Druckbereich. Damit lässt sich der optische Eindruck aus den Plots quantitativ bestätigen.

In [ ]:
temp_bins = pd.cut(df['temperature'], bins=[100, 150, 200, 250, 300], include_lowest=True)
df.groupby(temp_bins, observed=True)[TARGET].agg(['count', 'mean', 'median', 'std', 'min'])

In [ ]:
pres_bins = pd.cut(df['pressure'], bins=[5, 10, 15, 20, 25], include_lowest=True)
df.groupby(pres_bins, observed=True)[TARGET].agg(['count', 'mean', 'median', 'std', 'min'])

## Zusammenfassung der Erkenntnisse
- Der Datensatz enthält 3957 Zeilen und 6 numerische Spalten, **ohne fehlende Werte und ohne Duplikate**.
- Nur **Temperatur** und **Druck** sind unabhängige Eingangsgrößen; die übrigen drei Merkmale lassen sich exakt daraus berechnen (T·P, T²+P³, T³−P²).
- Die Zielvariable `quality_rating` ist **stark linksschief**: Die meisten Werte liegen nahe 100, nur wenige sind deutlich schlechter.
- Die **Temperatur** ist der entscheidende Einflussfaktor: Bis ca. 200–250 °C ist die Qualität praktisch perfekt, oberhalb von ca. 250 °C fällt sie stark ab. Der **Druck** hat kaum Einfluss.
- Der Zusammenhang ist **nichtlinear** → lineare Regression auf den Rohmerkmalen wird voraussichtlich nicht ausreichen.

## Nächste Schritte
- Umgang mit Multikollinearität (z. B. nur T und P verwenden oder Regularisierung)
- Feature Engineering (z. B. polynomiale Merkmale, Schwellwert-Merkmale für die Temperatur)
- Skalierung der Merkmale (z. B. `StandardScaler`)
- Ggf. Transformation der Zielvariable aufgrund der Schiefe
- Train/Test-Split (ggf. stratifiziert nach Temperaturbereich)
- Modellvorbereitung: Vergleich linearer und nichtlinearer Modelle (z. B. Polynomregression, Random Forest, Gradient Boosting)